In [6]:
import re
import hashlib
import requests
from bs4 import BeautifulSoup
import pandas as pd
from urllib.parse import urlparse

SESSION = requests.Session()
SESSION.headers.update({
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
})

def fetch_page(url):
    """Fetch a ufcstats.com page, solving the browser challenge if needed."""
    response = SESSION.get(url)
    if 'Checking your browser' not in response.text:
        return response

    match = re.search(r'var nonce="([^"]+)"', response.text)
    if not match:
        return response

    nonce = match.group(1)
    n = 0
    while not hashlib.sha256(f'{nonce}:{n}'.encode()).hexdigest().startswith('00'):
        n += 1

    base = f"{urlparse(url).scheme}://{urlparse(url).netloc}"
    SESSION.post(
        f'{base}/__c',
        data={'nonce': nonce, 'n': n},
        headers={'Content-Type': 'application/x-www-form-urlencoded'},
    )
    return SESSION.get(url)

url = "http://ufcstats.com/statistics/events/completed?page=all"
response = fetch_page(url)
response.raise_for_status()
soup = BeautifulSoup(response.content, 'html.parser')

# gets the link for each ufc event
event_links = []
for link in soup.select('a.b-link.b-link_style_black'):
    event_url = link.get('href')
    if event_url and 'event-details' in event_url:
        event_links.append(event_url)

print(f"Found {len(event_links)} events")


Found 776 events


In [7]:
fight_links = []

for event_url in event_links: #gets the links of all the matches from each ufc event
    response = fetch_page(event_url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Extract event date
    date_element = soup.find('li', class_='b-list__box-list-item')
    event_date = date_element.text.split(':')[-1].strip() if date_element else 'N/A'
    
    # Find all fight rows
    for row in soup.find_all('tr', {'class': 'b-fight-details__table-row'}):
        onclick = row.get('onclick', '')
        if 'doNav' in onclick:
            fight_url = onclick.split("'")[1]
            fight_links.append({
                'url': fight_url,
                'event_date': event_date  # Store date with fight URL
            })

print(f"Found {len(fight_links)} fights")



Found 8726 fights


In [8]:
def scrape_ufc_fight_details(url): # scrapes the stats from the url we input
    response = fetch_page(url)
    if 'Checking your browser' in response.text:
        raise RuntimeError('Browser challenge failed for ' + url)

    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Extract method of victory
    method_tag = soup.find('i', class_='b-fight-details__text-item_first')
    method = method_tag.find('i', style="font-style: normal").text.strip() if method_tag else ''
    
    # Extract referee
    referee = ""
    referee_span = soup.select_one('i.b-fight-details__label:-soup-contains("Referee:") + span')
    if referee_span:
        referee = referee_span.text.strip()

    # Find fighter status (W/L) for the first fighter
    fighter_status_tag = soup.find('i', class_='b-fight-details__person-status')
    first_fighter_won = None
    if fighter_status_tag:
        status = fighter_status_tag.text.strip()
        first_fighter_won = 1 if status == "W" else 0

    # get the data from round 1 and the totals rows
    all_rows = []
    rows = soup.find_all('tr', class_='b-fight-details__table-row')

    for row in rows:
        columns = row.find_all('p', class_='b-fight-details__table-text')
        data = [col.get_text(strip=True) for col in columns]
        all_rows.append(data)

    # Table 1 has 20-col rows (totals + per-round), table 2 has 18-col rows (sig strike breakdown)
    stat_rows_20 = [row for row in all_rows if len(row) == 20 and row[0] and row[1]]
    stat_rows_18 = [row for row in all_rows if len(row) == 18 and row[0] and row[1]]
    if len(stat_rows_20) < 2 or len(stat_rows_18) < 2:
        raise ValueError(
            f'Could not parse fight stats from {url} '
            f'(20-col: {len(stat_rows_20)}, 18-col: {len(stat_rows_18)})'
        )

    filtered_rows = [
        stat_rows_20[0],  # totals
        stat_rows_20[1],  # round 1 totals
        stat_rows_18[0],  # significant strike totals
        stat_rows_18[1],  # round 1 significant strikes
    ]

    filtered_rows.append([method, referee])
    filtered_rows.append(first_fighter_won)

    return filtered_rows

# Test with the example URL
url = 'http://ufcstats.com/fight-details/d05cb4c4135ce402'
data = scrape_ufc_fight_details(url)
print(data)


[['Michael Chandler', 'Paddy Pimblett', '0', '0', '11 of 28', '80 of 143', '39%', '55%', '20 of 39', '121 of 197', '4 of 7', '1 of 2', '57%', '50%', '0', '0', '0', '1', '2:44', '4:41'], ['Michael Chandler', 'Paddy Pimblett', '0', '0', '5 of 11', '16 of 40', '45%', '40%', '7 of 14', '25 of 50', '2 of 3', '0 of 0', '66%', '---', '0', '0', '0', '0', '2:10', '0:00'], ['Michael Chandler', 'Paddy Pimblett', '11 of 28', '80 of 143', '39%', '55%', '6 of 18', '61 of 119', '3 of 7', '4 of 5', '2 of 3', '15 of 19', '10 of 24', '43 of 96', '0 of 0', '4 of 5', '1 of 4', '33 of 42'], ['Michael Chandler', 'Paddy Pimblett', '5 of 11', '16 of 40', '45%', '40%', '4 of 10', '7 of 28', '1 of 1', '1 of 2', '0 of 0', '8 of 10', '4 of 7', '16 of 40', '0 of 0', '0 of 0', '1 of 4', '0 of 0'], ['KO/TKO', 'Kerry Hatley'], 0]


In [9]:
def format_fight_data_single_row(data, verbose=False): #formats the data extracted from a match into a dictionary
    if not data or len(data) < 4:
        return "Not enough data to format."
    
    # Extract fighter names for reference
    fighter1 = data[0][0]
    fighter2 = data[0][1]
    
    # Create a single row dictionary with p1_ and p2_ prefixes
    formatted_row = {
        "p1_fighter": fighter1,
        "p2_fighter": fighter2,
        
        # Totals (Row 1)
        "p1_KD": data[0][2],
        "p2_KD": data[0][3],
        "p1_SIG_STR": data[0][4],
        "p2_SIG_STR": data[0][5],
        "p1_SIG_STR_PCT": data[0][6],
        "p2_SIG_STR_PCT": data[0][7],
        "p1_TOTAL_STR": data[0][8],
        "p2_TOTAL_STR": data[0][9],
        "p1_TD": data[0][10],
        "p2_TD": data[0][11],
        "p1_TD_PCT": data[0][12],
        "p2_TD_PCT": data[0][13],
        "p1_SUB_ATT": data[0][14],
        "p2_SUB_ATT": data[0][15],
        "p1_REV": data[0][16],
        "p2_REV": data[0][17],
        "p1_CTRL": data[0][18],
        "p2_CTRL": data[0][19],
        
        # Round 1 Totals (Row 2)
        "p1_R1_KD": data[1][2],
        "p2_R1_KD": data[1][3],
        "p1_R1_SIG_STR": data[1][4],
        "p2_R1_SIG_STR": data[1][5],
        "p1_R1_SIG_STR_PCT": data[1][6],
        "p2_R1_SIG_STR_PCT": data[1][7],
        "p1_R1_TOTAL_STR": data[1][8],
        "p2_R1_TOTAL_STR": data[1][9],
        "p1_R1_TD": data[1][10],
        "p2_R1_TD": data[1][11],
        "p1_R1_TD_PCT": data[1][12],
        "p2_R1_TD_PCT": data[1][13],
        "p1_R1_SUB_ATT": data[1][14],
        "p2_R1_SUB_ATT": data[1][15],
        "p1_R1_REV": data[1][16],
        "p2_R1_REV": data[1][17],
        "p1_R1_CTRL": data[1][18],
        "p2_R1_CTRL": data[1][19],
        
        # Significant Strike Details (Row 3)
        "p1_SIG_STR_LANDED_ATTEMPTED": data[2][2],
        "p2_SIG_STR_LANDED_ATTEMPTED": data[2][3],
        "p1_SIG_STR_PCT_DETAILED": data[2][4],
        "p2_SIG_STR_PCT_DETAILED": data[2][5],
        "p1_HEAD_LANDED_ATTEMPTED": data[2][6],
        "p2_HEAD_LANDED_ATTEMPTED": data[2][7],
        "p1_BODY_LANDED_ATTEMPTED": data[2][8],
        "p2_BODY_LANDED_ATTEMPTED": data[2][9],
        "p1_LEG_LANDED_ATTEMPTED": data[2][10],
        "p2_LEG_LANDED_ATTEMPTED": data[2][11],
        "p1_DISTANCE_LANDED_ATTEMPTED": data[2][12],
        "p2_DISTANCE_LANDED_ATTEMPTED": data[2][13],
        "p1_CLINCH_LANDED_ATTEMPTED": data[2][14],
        "p2_CLINCH_LANDED_ATTEMPTED": data[2][15],
        "p1_GROUND_LANDED_ATTEMPTED": data[2][16],
        "p2_GROUND_LANDED_ATTEMPTED": data[2][17],
        
        # Round 1 Significant Strike Details (Row 4)
        "p1_R1_SIG_STR_LANDED_ATTEMPTED": data[3][2],
        "p2_R1_SIG_STR_LANDED_ATTEMPTED": data[3][3],
        "p1_R1_SIG_STR_PCT_DETAILED": data[3][4],
        "p2_R1_SIG_STR_PCT_DETAILED": data[3][5],
        "p1_R1_HEAD_LANDED_ATTEMPTED": data[3][6],
        "p2_R1_HEAD_LANDED_ATTEMPTED": data[3][7],
        "p1_R1_BODY_LANDED_ATTEMPTED": data[3][8],
        "p2_R1_BODY_LANDED_ATTEMPTED": data[3][9],
        "p1_R1_LEG_LANDED_ATTEMPTED": data[3][10],
        "p2_R1_LEG_LANDED_ATTEMPTED": data[3][11],
        "p1_R1_DISTANCE_LANDED_ATTEMPTED": data[3][12],
        "p2_R1_DISTANCE_LANDED_ATTEMPTED": data[3][13],
        "p1_R1_CLINCH_LANDED_ATTEMPTED": data[3][14],
        "p2_R1_CLINCH_LANDED_ATTEMPTED": data[3][15],
        "p1_R1_GROUND_LANDED_ATTEMPTED": data[3][16],
        "p2_R1_GROUND_LANDED_ATTEMPTED": data[3][17],

        "method": data[4][0],
        "referee": data[4][1],
        "winner": data[5]
    }
    
    if verbose:
        print(",".join(formatted_row.keys()))
        print(",".join(str(val) for val in formatted_row.values()))
    
    return formatted_row

# Call the function with your scraped data
formatted_row = format_fight_data_single_row(data, verbose=True)
print(formatted_row)


p1_fighter,p2_fighter,p1_KD,p2_KD,p1_SIG_STR,p2_SIG_STR,p1_SIG_STR_PCT,p2_SIG_STR_PCT,p1_TOTAL_STR,p2_TOTAL_STR,p1_TD,p2_TD,p1_TD_PCT,p2_TD_PCT,p1_SUB_ATT,p2_SUB_ATT,p1_REV,p2_REV,p1_CTRL,p2_CTRL,p1_R1_KD,p2_R1_KD,p1_R1_SIG_STR,p2_R1_SIG_STR,p1_R1_SIG_STR_PCT,p2_R1_SIG_STR_PCT,p1_R1_TOTAL_STR,p2_R1_TOTAL_STR,p1_R1_TD,p2_R1_TD,p1_R1_TD_PCT,p2_R1_TD_PCT,p1_R1_SUB_ATT,p2_R1_SUB_ATT,p1_R1_REV,p2_R1_REV,p1_R1_CTRL,p2_R1_CTRL,p1_SIG_STR_LANDED_ATTEMPTED,p2_SIG_STR_LANDED_ATTEMPTED,p1_SIG_STR_PCT_DETAILED,p2_SIG_STR_PCT_DETAILED,p1_HEAD_LANDED_ATTEMPTED,p2_HEAD_LANDED_ATTEMPTED,p1_BODY_LANDED_ATTEMPTED,p2_BODY_LANDED_ATTEMPTED,p1_LEG_LANDED_ATTEMPTED,p2_LEG_LANDED_ATTEMPTED,p1_DISTANCE_LANDED_ATTEMPTED,p2_DISTANCE_LANDED_ATTEMPTED,p1_CLINCH_LANDED_ATTEMPTED,p2_CLINCH_LANDED_ATTEMPTED,p1_GROUND_LANDED_ATTEMPTED,p2_GROUND_LANDED_ATTEMPTED,p1_R1_SIG_STR_LANDED_ATTEMPTED,p2_R1_SIG_STR_LANDED_ATTEMPTED,p1_R1_SIG_STR_PCT_DETAILED,p2_R1_SIG_STR_PCT_DETAILED,p1_R1_HEAD_LANDED_ATTEMPTED,p2_R1_HEAD_LAN

In [10]:
def process_multiple_fights_with_date(url_date_list): # gets the data from every match and creates a dataframe from it 
    all_fights_data = []
    
    for i, item in enumerate(url_date_list, 1):
        url = item['url']
        event_date = item['event_date']
        try:
            fight_data = scrape_ufc_fight_details(url)
            formatted_row = format_fight_data_single_row(fight_data, verbose=False)
            if isinstance(formatted_row, dict):
                formatted_row['event_date'] = event_date
                all_fights_data.append(formatted_row)
            else:
                print(f"Error processing {url}: {formatted_row}")
        except Exception as e:
            print(f"Error processing {url}: {str(e)}")

        if i % 50 == 0:
            print(f"Processed {i}/{len(url_date_list)} fights...")
    
    return pd.DataFrame(all_fights_data) if all_fights_data else pd.DataFrame()

fights_df = process_multiple_fights_with_date(fight_links)
print(f"Scraped {len(fights_df)} fights with {len(fights_df.columns)} columns")
print(fights_df.head())
fights_df.to_csv('ufc_fights_data.csv', index=False)
print("Saved ufc_fights_data.csv")


Processed 50/8726 fights...
Processed 100/8726 fights...
Processed 150/8726 fights...
Processed 200/8726 fights...
Processed 250/8726 fights...
Processed 300/8726 fights...
Processed 350/8726 fights...
Processed 400/8726 fights...
Processed 450/8726 fights...
Processed 500/8726 fights...
Processed 550/8726 fights...
Processed 600/8726 fights...
Processed 650/8726 fights...
Processed 700/8726 fights...
Processed 750/8726 fights...
Processed 800/8726 fights...
Processed 850/8726 fights...
Processed 900/8726 fights...
Processed 950/8726 fights...
Processed 1000/8726 fights...
Processed 1050/8726 fights...
Processed 1100/8726 fights...
Processed 1150/8726 fights...
Processed 1200/8726 fights...
Processed 1250/8726 fights...
Processed 1300/8726 fights...
Processed 1350/8726 fights...
Processed 1400/8726 fights...
Processed 1450/8726 fights...
Processed 1500/8726 fights...
Processed 1550/8726 fights...
Processed 1600/8726 fights...
Processed 1650/8726 fights...
Processed 1700/8726 fights...
